# Race Time Predictor

Prédiction du temps de course à partir d'une trace GPX,
basée sur le modèle de coût métabolique de Minetti (2002).

**Modèles intégrés**
- Minetti (2002) : coût métabolique en fonction de la pente
- Fatigue exponentielle à plateau (calibrée sur historique athlète)
- Plafond vitesse en descente
- Dégradation thermique (Ely et al. 2007) avec correction altitudinale (Rolland 2003)


Prédire son temps sur un trail, c'est tenter de réconcilier la physique de l'effort avec la réalité du terrain. Le modèle de Minetti (2002) offre une base biomécanique solide : il relie le coût métabolique de la locomotion à la pente, en J/(kg·m). Sur terrain plat, ce coût est d'environ 3,6 J/(kg·m). En montée à 20%, il bondit à plus de 10 J/(kg·m) — soit une vitesse théorique divisée par presque 3.

Mais Minetti est un modèle *stationnaire* : il ignore la fatigue qui s'accumule, la chaleur qui s'installe, et les limites mécaniques propres aux descentes techniques. Ce notebook empile trois corrections successives sur le modèle de base, chacune ancrée dans la littérature scientifique.


In [1]:
import sys
import warnings
import requests
warnings.filterwarnings('ignore')

sys.path.insert(0, '../core')

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from gpx_race import parse_gpx, segment_trace
from race_predictor import (
    race_summary, format_time, format_pace,
    minetti_speed_ratio, minetti_cost, minetti_cost_flat
)


## 1. Paramètres

In [2]:
# --- Trace GPX ---
GPX_PATH  = '/home/gsainton/Downloads/ultra-trail-du-haut-giffre-2026-trail-du-mont-blanc-des-dames.gpx'
RACE_NAME = 'Trail du Mont-Blanc des Dames 2026'
RACE_DATE = '2026-06-21'   # YYYY-MM-DD — utilisé pour la météo forecast

# --- Heure de départ ---
START_TIME = '04:00'  # HH:MM

# --- Vitesse de référence sur terrain plat ---
V_FLAT_KMH    = 9   # km/h
ATHLETE_FACTOR = 0.85  # < 1.0 = terrain technique / prudence

# --- Segmentation RDP ---
EPSILON_M  = 1.0   # tolérance altitude (m)
MIN_SEG_KM = 0.05  # longueur minimale d'un segment (km)

# --- Modèle de fatigue (calibré sur EcoTrail, Imperial, SainteLyon) ---

FATIGUE_A = 0.753
FATIGUE_K = 2.981


USE_FATIGUE = True   # False = Minetti pur

# --- Plafond vitesse en descente ---
V_DOWNHILL_MAX_KMH = 9.0

# --- Modèle thermique (Ely et al. 2007 / Rolland 2003) ---
T_THRESHOLD     = 15.0   # °C — seuil de dégradation
ALPHA_HEAT      = 0.004  # 0.4 % de dégradation par degré au-dessus du seuil
TEMP_LAPSE_RATE = 0.0065 # °C/m — gradient altitudinal standard
USE_THERMAL     = True   # False = sans effet thermique

# --- Ravitaillements : (nom, distance km) ---
RAVITOS = [
    ('Joux Plane',    18.3),
    ('Golèse',        30.0),
    ('Refuge Bostan', 38.1),
    ('Vallon',        46.6),
    ('Sixt',          59.2),
    ('Arrivée',       71.3),
]

# --- Barrières horaires : (nom, distance km, heure HH:MM ou None) ---
BARRIERES = [
    ('Joux Plane',    18.3, '8:00'),
    ('Golèse',        30.0, None),
    ('Refuge Bostan', 38.1, None),
    ('Vallon',        46.6, '16:00'),
    ('Sixt',          59.2, '20:00'),
    ('Arrivée',       71.3, '23:30'),
]


## 2. Fonctions utilitaires

### Le modèle de Minetti (2002)

Le cœur du modèle,  c'est le polynôme des mesures de consommation d'oxygène à différentes pentes :

$$C(i) = 155.4 \cdot i^5 - 30.4 \cdot i^4 - 43.3 \cdot i^3 + 46.3 \cdot i^2 + 19.5 \cdot i + 3.6$$

avec $i$ la pente en fraction (−0.45 à +0.45), et $C$ en J/(kg·m).

La vitesse sur un segment de pente $i$ est estimée par rapport à la vitesse de référence sur terrain plat $V_0$ :

$$V(i) = V_0 \cdot \frac{C(0)}{C(i)}$$

**Paramètre clé** : `V_FLAT_KMH` — vitesse de course sur terrain plat (from mon historique).

**Limite connue** : Minetti prédit des vitesses de descente élevées (jusqu'à 20+ km/h pour les pentes à −15%). J'ajoute donc un plafond `V_DOWNHILL_MAX_KMH`.

> Minetti AE et al. (2002). *Energy cost of walking and running at extreme uphill and downhill slopes*. J Appl Physiol 93(3):1039-1046.


In [3]:
def parse_hhmm(hhmm):
    """Convert HH:MM string to minutes since midnight."""
    h, m = map(int, hhmm.split(':'))
    return h * 60 + m


def plateau_fatigue(x, a, k):
    """
    Exponential fatigue model with plateau.

    x : fraction de distance parcourue [0, 1]
    a : facteur asymptotique (plateau en fin de course)
    k : vitesse de dégradation initiale
    """
    return a + (1 - a) * np.exp(-k * x)


def interpolate_temp_at_time(df_weather, time_minutes_since_midnight):
    """Interpolate temperature (°C) from hourly forecast at a given time."""
    hours = df_weather['time'].dt.hour + df_weather['time'].dt.minute / 60
    temp  = df_weather['temperature_2m'].values
    return float(np.interp(time_minutes_since_midnight / 60, hours, temp))


def altitudinal_correction(temp_ref, alt_ref, alt_target,
                            lapse_rate=TEMP_LAPSE_RATE):
    """Apply standard lapse rate correction to temperature."""
    return temp_ref - lapse_rate * (alt_target - alt_ref)


def thermal_factor(temp_c, t_threshold=T_THRESHOLD, alpha=ALPHA_HEAT):
    """Speed degradation factor due to heat (Ely et al. 2007)."""
    return max(0.0, 1.0 - alpha * max(0.0, temp_c - t_threshold))


def predict_segments_full(df_segments, v_flat_kmh, athlete_factor,
                          fatigue_a, fatigue_k, use_fatigue,
                          v_downhill_max_kmh, start_min,
                          df_weather, alt_ref, use_thermal):
    """
    Predict speed and time per segment.

    Combines Minetti cost model with fatigue, downhill cap, and thermal effect.
    """
    total_dist = df_segments['length_km'].sum()
    rows  = []
    t_cum = 0.0
    d_cum = 0.0

    for _, seg in df_segments.iterrows():
        dist_pct = (d_cum + seg['length_km'] / 2) / total_dist
        alt_mid  = (seg['ele_start'] + seg['ele_end']) / 2

        fatigue = plateau_fatigue(dist_pct, fatigue_a, fatigue_k) if use_fatigue else 1.0

        heat, temp_seg = 1.0, None
        if use_thermal and df_weather is not None:
            temp_ref = interpolate_temp_at_time(df_weather, start_min + t_cum)
            temp_seg = altitudinal_correction(temp_ref, alt_ref, alt_mid)
            heat     = thermal_factor(temp_seg)

        ratio  = minetti_speed_ratio(seg['slope_mean_pct'])
        v_pred = v_flat_kmh * ratio * athlete_factor * fatigue * heat

        if seg['slope_mean_pct'] < -3:
            v_pred = min(v_pred, v_downhill_max_kmh)
        v_pred = max(v_pred, 1.0)

        pace  = 60.0 / v_pred
        t_min = seg['length_km'] / v_pred * 60.0
        t_cum += t_min
        d_cum += seg['length_km']

        rows.append({
            **seg.to_dict(),
            'speed_kmh':       round(v_pred, 2),
            'pace_min_per_km': round(pace, 2),
            'time_min':        round(t_min, 1),
            'time_cum_min':    round(t_cum, 1),
            'effort_index':    round(minetti_cost(seg['slope_mean_pct']) / minetti_cost_flat(), 2),
            'fatigue_factor':  round(fatigue, 3),
            'temp_seg_c':      round(temp_seg, 1) if temp_seg is not None else None,
            'thermal_factor':  round(heat, 3),
        })

    return pd.DataFrame(rows)


## 3. Météo forecast

### Correction thermique

Au-delà d'un seuil de température, la performance en endurance se dégrade. Ely et al. (2007, 2008) ont quantifié cet effet sur des données de marathon : environ **0,4% de perte de vitesse par degré Celsius** au-dessus de 10°C (WBGT). On retient ici 15°C comme seuil, plus adapté à un effort d'ultra longue durée.

$$f_{\text{therm}}(T) = 1 - \alpha \cdot \max(0, T - T_{\text{seuil}})$$

En plus, la température n'est pas uniforme le long du parcours : elle varie avec l'altitude. On applique le gradient altitudinal standard de Rolland (2003) :

$$T_{\text{segment}} = T_{\text{ref}}(t) - \Gamma \cdot (z_{\text{segment}} - z_{\text{ref}})$$

avec $\Gamma = 0.0065$ °C/m et $T_{\text{ref}}(t)$ la température interpolée depuis la prévision Open-Meteo à l'heure de passage sur ce segment.

Le **WBGT** (Wet Bulb Globe Temperature) est calculé selon la formule simplifiée de Bernard & Kenney (1994), combinant température et humidité relative. C'est l'indice de référence en médecine du sport pour caractériser le stress thermique.

> Ely BR et al. (2007). *Impact of weather on marathon-running performance*. Med Sci Sports Exerc 39(3):487-493.  
> Rolland C (2003). *Spatial and seasonal variations of air temperature lapse rates in Alpine regions*. J Climate 16:1032-1046.


In [4]:
_FORECAST_URL = 'https://api.open-meteo.com/v1/forecast'
_HOURLY_VARS  = [
    'temperature_2m', 'relative_humidity_2m', 'apparent_temperature',
    'precipitation', 'wind_speed_10m', 'shortwave_radiation',
]


def fetch_weather_forecast(lat, lon, date_str, timezone='Europe/Paris', date_end=None):
    """
    Fetch hourly weather forecast from Open-Meteo (up to 16 days ahead).

    Returns pd.DataFrame or None on error.
    """
    params = {
        'latitude':        lat,
        'longitude':       lon,
        'start_date':      date_str,
        'end_date':        date_end or date_str,
        'hourly':          ','.join(_HOURLY_VARS),
        'wind_speed_unit': 'kmh',
        'timezone':        timezone,
    }
    try:
        resp = requests.get(_FORECAST_URL, params=params, timeout=10)
        resp.raise_for_status()
        hourly = resp.json().get('hourly', {})
    except Exception as e:
        print(f'Erreur Open-Meteo forecast : {e}')
        return None

    df = pd.DataFrame({
        'time':                 pd.to_datetime(hourly['time']),
        'temperature_2m':       hourly['temperature_2m'],
        'relative_humidity_2m': hourly['relative_humidity_2m'],
        'apparent_temperature': hourly['apparent_temperature'],
        'precipitation':        hourly['precipitation'],
        'wind_speed_10m':       hourly['wind_speed_10m'],
        'shortwave_radiation':  hourly['shortwave_radiation'],
    })
    df = df[df['time'].dt.date == pd.to_datetime(date_str).date()].reset_index(drop=True)

    # WBGT simplifié (Bernard & Kenney 1994)
    df['wbgt'] = (
        0.567 * df['temperature_2m']
        + 0.393 * (df['relative_humidity_2m'] / 100 * 6.105
                   * np.exp(17.27 * df['temperature_2m'] / (237.7 + df['temperature_2m'])))
        + 3.94
    )
    return df


# Centroïde de la trace
df_gpx   = parse_gpx(GPX_PATH)
df_seg   = segment_trace(df_gpx, epsilon_m=EPSILON_M, min_seg_km=MIN_SEG_KM)
start_min = parse_hhmm(START_TIME)
lat_c    = df_gpx['lat'].mean()
lon_c    = df_gpx['lon'].mean()
alt_ref  = float(df_gpx['ele'].mean())

df_weather = fetch_weather_forecast(lat_c, lon_c, RACE_DATE)

if df_weather is not None:
    print(f'Prévisions pour le {RACE_DATE} — centroïde ({lat_c:.3f}N, {lon_c:.3f}E)')
    print(df_weather[['time', 'temperature_2m', 'relative_humidity_2m', 'wbgt']].to_string(index=False))
else:
    print('Prévision indisponible — effet thermique désactivé.')
    USE_THERMAL = False


Prévisions pour le 2026-06-21 — centroïde (46.104N, 6.739E)
               time  temperature_2m  relative_humidity_2m      wbgt
2026-06-21 00:00:00            16.9                    66 18.505170
2026-06-21 01:00:00            16.8                    63 18.191937
2026-06-21 02:00:00            16.7                    61 17.956271
2026-06-21 03:00:00            16.3                    61 17.615371
2026-06-21 04:00:00            15.9                    62 17.347835
2026-06-21 05:00:00            16.0                    62 17.432651
2026-06-21 06:00:00            17.1                    62 18.376195
2026-06-21 07:00:00            18.6                    62 19.695426
2026-06-21 08:00:00            20.2                    61 21.054044
2026-06-21 09:00:00            21.8                    58 22.237864
2026-06-21 10:00:00            23.4                    54 23.298287
2026-06-21 11:00:00            24.8                    50 24.134202
2026-06-21 12:00:00            26.0                    4

## 4. Prédiction par segment

### Les trois corrections empilées

La vitesse finale sur chaque segment résulte de la multiplication de quatre facteurs :

$$V_{\text{prédit}} = V_0 \cdot r_{\text{Minetti}}(i) \cdot f_{\text{athlète}} \cdot f_{\text{fatigue}}(x) \cdot f_{\text{therm}}(T)$$

| Facteur | Rôle | Paramètre |
|---|---|---|
| $r_{\text{Minetti}}(i)$ | Ralentissement/accélération lié à la pente | `slope_mean_pct` |
| $f_{\text{athlète}}$ | Correction globale terrain technique, style | `ATHLETE_FACTOR` |
| $f_{\text{fatigue}}(x)$ | Dégradation progressive au fil de la distance | `FATIGUE_A`, `FATIGUE_K` |
| $f_{\text{therm}}(T)$ | Dégradation liée à la chaleur | `T_THRESHOLD`, `ALPHA_HEAT` |

**Le modèle de fatigue** suit une exponentielle à plateau, calibrée sur trois courses de référence (EcoTrail Paris, Imperial Fontainebleau, SainteLyon).

$$f_{\text{fatigue}}(x) = a + (1-a) \cdot e^{-k \cdot x}$$

avec $x \in [0, 1]$ la fraction de distance parcourue.


In [5]:
df_pred = predict_segments_full(
    df_seg,
    v_flat_kmh=V_FLAT_KMH,
    athlete_factor=ATHLETE_FACTOR,
    fatigue_a=FATIGUE_A,
    fatigue_k=FATIGUE_K,
    use_fatigue=USE_FATIGUE,
    v_downhill_max_kmh=V_DOWNHILL_MAX_KMH,
    start_min=start_min,
    df_weather=df_weather,
    alt_ref=alt_ref,
    use_thermal=USE_THERMAL,
)

summary = race_summary(df_pred, race_name=RACE_NAME)
end_min = start_min + summary['total_min']

print(f"Course   : {summary['race_name']}")
print(f"Distance : {summary['distance_km']} km  |  D+ : {summary['dplus_m']} m  |  D- : {summary['dmoins_m']} m")
print(f"Segments : {summary['n_segments']}")
print(f"Temps    : {summary['total_time_str']}  (allure moy. {summary['avg_pace_str']} /km)")
print(f"Arrivée  : {int(end_min)//60 % 24:02d}h{int(end_min)%60:02d}")


Course   : Trail du Mont-Blanc des Dames 2026
Distance : 71.28 km  |  D+ : 4840 m  |  D- : 4855 m
Segments : 29
Temps    : 14h 57min  (allure moy. 12'35" /km)
Arrivée  : 18h57


In [6]:
def plot_temperature_forecast(df_weather, start_min, total_min, race_name):
    """Plot hourly temperature forecast with race window highlighted."""
    if df_weather is None:
        print("Pas de données météo disponibles.")
        return

    hours = df_weather['time'].dt.hour + df_weather['time'].dt.minute / 60
    temp  = df_weather['temperature_2m'].values
    wbgt  = df_weather['wbgt'].values

    t_start_h = start_min / 60
    t_end_h   = (start_min + total_min) / 60

    fig = go.Figure()

    fig.add_vrect(
        x0=t_start_h, x1=t_end_h,
        fillcolor='rgba(52,152,219,0.12)', line_width=0,
        annotation_text='Fenêtre de course', annotation_position='top left',
        annotation_font=dict(color='#3498db', size=10),
    )
    fig.add_trace(go.Scatter(
        x=hours, y=temp, mode='lines+markers',
        name='Température 2m (°C)',
        line=dict(color='#e67e22', width=2.5), marker=dict(size=5),
        hovertemplate='%{x:.1f}h : %{y:.1f}°C<extra></extra>',
    ))
    fig.add_trace(go.Scatter(
        x=hours, y=wbgt, mode='lines',
        name='WBGT (°C)',
        line=dict(color='#e74c3c', width=1.5, dash='dot'),
        hovertemplate='%{x:.1f}h : WBGT=%{y:.1f}°C<extra></extra>',
    ))
    fig.add_hline(
        y=T_THRESHOLD, line=dict(color='#f1c40f', width=1, dash='dash'),
        annotation_text=f'Seuil dégradation ({T_THRESHOLD}°C)',
        annotation_position='right',
        annotation_font=dict(color='#f1c40f', size=9),
    )
    fig.update_layout(
        template='plotly_dark',
        title=f'Prévision météo journalière — {race_name}',
        xaxis=dict(
            title='Heure de la journée',
            tickvals=list(range(0, 25, 2)),
            ticktext=[f'{h:02d}h00' for h in range(0, 25, 2)],
        ),
        yaxis=dict(title='Température (°C)'),
        height=350, margin=dict(t=60, b=50, l=60, r=20),
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
    )
    fig.show()


plot_temperature_forecast(df_weather, start_min, df_pred['time_min'].sum(), RACE_NAME)

## 5. Profil altimétrique

### Lire le profil

Le profil ci-dessous est coloré selon la catégorie de pente de chaque segment. En survolant un segment, le tooltip affiche le facteur de fatigue et la température effective à ce point du parcours.

Les **lignes jaunes pointillées** marquent les ravitaillements, avec l'heure de passage prédite. Les **lignes rouges en tirets** signalent les barrières horaires, avec la marge positive ou négative.

Le panneau inférieur montre l'évolution de la température corrigée de l'altitude et du WBGT le long du parcours.


In [7]:
from plotly.subplots import make_subplots

SLOPE_COLORS = {
    'montée raide':   '#e74c3c',
    'montée':         '#e67e22',
    'plat':           '#2ecc71',
    'descente':       '#3498db',
    'descente raide': '#9b59b6',
}


def make_elevation_profile(df_gpx, df_pred, ravitos, barrieres, start_min,
                            df_weather=None):
    """
    Build interactive elevation profile with temperature subplot.

    Top panel   : altitude colored by slope category, ravitos, cutoffs.
    Bottom panel: temperature corrected for altitude and WBGT along the route.
    """
    has_temp = df_weather is not None and 'temp_seg_c' in df_pred.columns

    row_heights = [0.70, 0.30] if has_temp else [1.0]
    n_rows      = 2 if has_temp else 1

    fig = make_subplots(
        rows=n_rows, cols=1,
        shared_xaxes=True,
        row_heights=row_heights,
        vertical_spacing=0.04,
    )

    dist_km = df_gpx['dist_cum'].values / 1000.0
    ele     = df_gpx['ele'].values

    # Fond altimétrique
    fig.add_trace(go.Scatter(
        x=dist_km, y=ele,
        fill='tozeroy', fillcolor='rgba(150,150,150,0.15)',
        line=dict(color='rgba(0,0,0,0)', width=0),
        showlegend=False, hoverinfo='skip',
    ), row=1, col=1)

    # Segments colorés par catégorie de pente
    legend_added = set()
    for _, seg in df_pred.iterrows():
        cat   = seg['slope_category']
        color = SLOPE_COLORS.get(cat, '#888')
        mask  = (dist_km >= seg['dist_start_km']) & (dist_km <= seg['dist_end_km'])
        show_legend = cat not in legend_added
        legend_added.add(cat)

        factors = f"fatigue={seg['fatigue_factor']:.2f}"
        if seg['temp_seg_c'] is not None:
            factors += f" | T={seg['temp_seg_c']:.1f}°C therm={seg['thermal_factor']:.2f}"

        fig.add_trace(go.Scatter(
            x=dist_km[mask], y=ele[mask],
            mode='lines', line=dict(color=color, width=2.5),
            name=cat, legendgroup=cat, showlegend=show_legend,
            hovertemplate=(
                f"<b>{cat}</b><br>"
                "Dist : %{x:.1f} km | Alt : %{y:.0f} m<br>"
                f"{factors}<extra></extra>"
            ),
        ), row=1, col=1)

    # Ravitaillements
    for nom, dist in ravitos:
        t_min = float(df_pred[df_pred['dist_end_km'] <= dist]['time_min'].sum())
        heure = (start_min + t_min) % (24 * 60)
        heure_str = f"{int(heure)//60:02d}h{int(heure)%60:02d}"
        fig.add_vline(x=dist, line=dict(color='#f1c40f', width=1.5, dash='dot'))
        fig.add_annotation(
            x=dist, y=max(ele) * 1.02, yref='y',
            text=f"<b>{nom}</b><br>{heure_str}",
            showarrow=False, font=dict(size=10, color='#f1c40f'),
            bgcolor='rgba(0,0,0,0.5)', bordercolor='#f1c40f', borderwidth=1,
            xanchor='center',
        )

    # Barrières horaires
    for barriere in barrieres:
        if barriere[2] is None:
            continue
        nom, dist, heure_limite = barriere
        marge_min = parse_hhmm(heure_limite) - (
            start_min + float(df_pred[df_pred['dist_end_km'] <= dist]['time_min'].sum())
        )
        signe = '+' if marge_min >= 0 else ''
        fig.add_vline(x=dist, line=dict(color='#e74c3c', width=1.5, dash='dash'))
        fig.add_annotation(
            x=dist, y=min(ele) * 0.85, yref='y',
            text=f"<b>Barr. {heure_limite}</b><br>{signe}{int(marge_min)}min",
            showarrow=False, font=dict(size=9, color='#e74c3c'),
            bgcolor='rgba(0,0,0,0.5)', bordercolor='#e74c3c', borderwidth=1,
            xanchor='center',
        )

    # Subplot température
    if has_temp:
        seg_dists = (df_pred['dist_start_km'] + df_pred['dist_end_km']) / 2
        seg_temps = df_pred['temp_seg_c'].values

        # Interpolation fine sur la grille GPX pour une courbe lisse
        dist_fine  = np.linspace(seg_dists.min(), seg_dists.max(), 500)
        temp_fine  = np.interp(dist_fine, seg_dists, seg_temps)

        # WBGT interpolé le long de la distance
        # On réutilise la correction alt pour chaque point GPX
        alt_ref_val = float(df_gpx['ele'].mean())
        hours_w = df_weather['time'].dt.hour + df_weather['time'].dt.minute / 60

        # Temps écoulé à chaque point GPX → température horaire → correction alt
        t_cum_gpx = np.zeros(len(df_gpx))
        speed_arr = np.interp(
            df_gpx['dist_cum'].values / 1000,
            (df_pred['dist_start_km'] + df_pred['dist_end_km']) / 2,
            df_pred['speed_kmh'].values,
        )
        speed_arr = np.where(speed_arr > 0, speed_arr, 1.0)
        ds = np.diff(df_gpx['dist_cum'].values / 1000, prepend=0)
        dt = ds / speed_arr * 60  # minutes
        t_cum_gpx = start_min + np.cumsum(dt)

        temp_gpx_ref = np.interp(t_cum_gpx / 60,
                                  hours_w.values,
                                  df_weather['temperature_2m'].values)
        temp_gpx_corr = temp_gpx_ref - TEMP_LAPSE_RATE * (df_gpx['ele'].values - alt_ref_val)

        # WBGT corrigé
        rh_gpx = np.interp(t_cum_gpx / 60,
                             hours_w.values,
                             df_weather['relative_humidity_2m'].values)
        wbgt_gpx = (
            0.567 * temp_gpx_corr
            + 0.393 * (rh_gpx / 100 * 6.105
                       * np.exp(17.27 * temp_gpx_corr / (237.7 + temp_gpx_corr)))
            + 3.94
        )

        fig.add_trace(go.Scatter(
            x=dist_km, y=temp_gpx_corr,
            mode='lines', name='T° corrigée altitude (°C)',
            line=dict(color='#f39c12', width=2),
            hovertemplate='%{x:.1f} km : %{y:.1f}°C<extra>T° corrigée</extra>',
        ), row=2, col=1)

        fig.add_trace(go.Scatter(
            x=dist_km, y=wbgt_gpx,
            mode='lines', name='WBGT corrigé (°C)',
            line=dict(color='#e74c3c', width=1.5, dash='dot'),
            hovertemplate='%{x:.1f} km : WBGT=%{y:.1f}°C<extra>WBGT</extra>',
        ), row=2, col=1)

        fig.add_hline(
            y=T_THRESHOLD,
            line=dict(color='#f1c40f', width=1, dash='dash'),
            annotation_text=f'Seuil {T_THRESHOLD}°C',
            annotation_position='bottom right',
            annotation_font=dict(color='#f1c40f', size=9),
            row=2, col=1,
        )

    fig.update_layout(
        template='plotly_dark',
        title=dict(text=f'Profil altimétrique — {RACE_NAME}', font=dict(size=16)),
        height=620 if has_temp else 500,
        margin=dict(t=80, b=50, l=60, r=20),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    )
    fig.update_xaxes(title_text='Distance (km)', row=n_rows, col=1,
                     showgrid=True, gridcolor='rgba(255,255,255,0.1)')
    fig.update_yaxes(title_text='Altitude (m)', row=1, col=1,
                     showgrid=True, gridcolor='rgba(255,255,255,0.1)')
    if has_temp:
        fig.update_yaxes(title_text='Température (°C)', row=2, col=1,
                         showgrid=True, gridcolor='rgba(255,255,255,0.08)')

    return fig


make_elevation_profile(df_gpx, df_pred, RAVITOS, BARRIERES, start_min, df_weather).show()


## 6. Table des segments

In [8]:
def style_segments(df):
    """Apply color styling to the segments table."""
    def color_slope(val):
        return {
            'montée raide':   'color: #e74c3c',
            'montée':         'color: #e67e22',
            'plat':           'color: #2ecc71',
            'descente':       'color: #3498db',
            'descente raide': 'color: #9b59b6',
        }.get(val, '')

    return (
        df.style
        .applymap(color_slope, subset=['Catégorie'])
        .set_properties(**{'text-align': 'center'})
        .set_table_styles([{'selector': 'th',
                            'props': [('text-align', 'center'), ('font-weight', 'bold')]}])
    )


t_cum  = start_min
heures = []
for _, seg in df_pred.iterrows():
    t_cum += seg['time_min']
    h = t_cum % (24 * 60)
    heures.append(f"{int(h)//60:02d}h{int(h)%60:02d}")

cols_display = [
    'dist_start_km', 'dist_end_km', 'length_km',
    'dplus_m', 'dmoins_m', 'slope_mean_pct', 'slope_category',
    'speed_kmh', 'pace_min_per_km', 'time_min',
    'effort_index', 'fatigue_factor', 'temp_seg_c', 'thermal_factor',
]

cols_display_watch = [
    'dist_start_km', 
    'dist_end_km', 
    'length_km',
    'dplus_m', 
    'dmoins_m', 
    'slope_mean_pct',
    'pace_min_per_km', 
    'speed_kmh',
    'time_min',
    #'temp_seg_c', 
]



#df_display = df_pred[cols_display].copy()
df_display = df_pred[cols_display_watch].copy()

df_display['pace_min_per_km'] = df_display['pace_min_per_km'].apply(format_pace)
df_display['time_min']        = df_display['time_min'].apply(format_time)
df_display['heure_arr']       = heures
df_display = df_display.rename(columns={
    'dist_start_km':  'De (km)',
    'dist_end_km':    'À (km)',
    'length_km':      'Dist (km)',
    'dplus_m':        'D+ (m)',
    'dmoins_m':       'D- (m)',
    'slope_mean_pct': 'Pente (%)',
    #'slope_category': 'Catégorie',
    #'speed_kmh':      'Vitesse (km/h)',
    'pace_min_per_km':'Allure (/km)',
    'time_min':       'Temps',
    #'effort_index':   'Effort',
    #'fatigue_factor': 'Fatigue',
    #'temp_seg_c':     'T° (°C)',
    #'thermal_factor': 'Therm.',
    #'heure_arr':      'Heure arr.',
})

#style_segments(df_display)
display(df_display) 

,De (km),À (km),Dist (km),D+ (m),D- (m),Pente (%),Allure (/km),speed_kmh,Temps,heure_arr
0,0.000000,1.271085,1.271085,0.293560,13.405361,-1.031544,"7'38""",7.86,9min,04h09
1,1.271085,1.736756,0.465671,10.769533,0.000000,2.312690,"9'13""",6.51,4min,04h14
2,1.736756,3.669185,1.932429,109.125531,2.093495,5.538731,"11'00""",5.45,21min,04h35
3,3.669185,6.701304,3.032119,407.312847,0.000000,13.433273,"16'07""",3.72,48min,05h24
4,6.701304,9.165880,2.464576,11.287022,178.272638,-6.775429,"6'40""",9.00,16min,05h40
5,9.165880,13.084866,3.918986,593.663289,0.516509,15.135210,"18'03""",3.32,1h 10min,06h51
6,13.084866,14.366802,1.281936,9.354037,78.373325,-5.383989,"6'40""",9.00,8min,06h59
7,14.366802,15.508542,1.141740,51.440667,0.207911,4.487252,"11'20""",5.29,13min,07h12
8,15.508542,16.363663,0.855120,197.311540,0.000000,23.074123,"25'04""",2.39,21min,07h34
9,16.363663,18.345949,1.982287,38.566525,172.124551,-6.737574,"6'40""",9.00,13min,07h47


## 7. Décomposition par tronçons

In [9]:
def build_splits_table(df_pred, ravitos, barrieres, start_min):
    """Build a splits table grouped by ravito sections."""
    bounds = [0.0] + [d for _, d in ravitos]
    names  = ['Départ'] + [n for n, _ in ravitos]
    barrieres_dict = {d: (n, h) for n, d, h in barrieres if h is not None}

    rows  = []
    t_cum = start_min

    for k in range(len(bounds) - 1):
        d_start = bounds[k]
        d_end   = bounds[k + 1]
        label   = f"{names[k]} → {names[k+1]}"

        mask = (
            (df_pred['dist_start_km'] < d_end + 0.01) &
            (df_pred['dist_end_km']   > d_start - 0.01)
        )
        sub = df_pred[mask]
        if sub.empty:
            continue

        t_seg = dp = dm = dist = 0.0
        for _, seg in sub.iterrows():
            frac   = (min(seg['dist_end_km'], d_end) - max(seg['dist_start_km'], d_start)) / seg['length_km']
            t_seg += seg['time_min'] * frac
            dp    += seg['dplus_m']  * frac
            dm    += seg['dmoins_m'] * frac
            dist  += seg['length_km'] * frac

        speed = dist / (t_seg / 60) if t_seg > 0 else 0
        pace  = 60 / speed if speed > 0 else None

        t_cum    += t_seg
        heure_arr = t_cum % (24 * 60)
        heure_str = f"{int(heure_arr)//60:02d}h{int(heure_arr)%60:02d}"

        barriere_str = ''
        if d_end in barrieres_dict:
            _, h_lim = barrieres_dict[d_end]
            marge    = parse_hhmm(h_lim) - t_cum
            signe    = '+' if marge >= 0 else ''
            barriere_str = f"{h_lim} ({signe}{int(marge)}min)"

        rows.append({
            'Tronçon':        label,
            'Dist (km)':      round(float(dist), 1),
            'D+ (m)':         int(dp),
            'D- (m)':         int(dm),
            'Temps':          format_time(t_seg),
            'Vitesse (km/h)': round(float(speed), 1),
            'Allure (/km)':   format_pace(pace),
            'Heure arr.':     heure_str,
            'Barrière':       barriere_str,
        })

    return pd.DataFrame(rows)


def style_splits(df):
    """Apply color styling to the splits table."""
    def color_barriere(val):
        if not val:
            return ''
        return 'color: #2ecc71; font-weight: bold' if '+' in val else 'color: #e74c3c; font-weight: bold'

    return (
        df.style
        .applymap(color_barriere, subset=['Barrière'])
        .set_properties(**{'text-align': 'center'})
        .set_table_styles([{'selector': 'th',
                            'props': [('text-align', 'center'), ('font-weight', 'bold')]}])
    )


df_splits = build_splits_table(df_pred, RAVITOS, BARRIERES, start_min)
print(f"Départ : {START_TIME} | V_flat : {V_FLAT_KMH} km/h | Facteur athlète : {ATHLETE_FACTOR}")
print(f"Fatigue : a={FATIGUE_A} k={FATIGUE_K} | Descente max : {V_DOWNHILL_MAX_KMH} km/h | Thermique : {USE_THERMAL}")
print()
style_splits(df_splits)


Départ : 04:00 | V_flat : 9 km/h | Facteur athlète : 0.85
Fatigue : a=0.753 k=2.981 | Descente max : 9.0 km/h | Thermique : True



,Tronçon,Dist (km),D+ (m),D- (m),Temps,Vitesse (km/h),Allure (/km),Heure arr.,Barrière
0,Départ → Joux Plane,18.300000,1428,441,3h 47min,4.800000,"12'24""",07h47,8:00 (+12min)
1,Joux Plane → Golèse,11.700000,1073,985,2h 36min,4.500000,"13'23""",10h23,
2,Golèse → Refuge Bostan,8.100000,652,747,1h 48min,4.500000,"13'23""",12h12,
3,Refuge Bostan → Vallon,8.500000,159,1022,1h 12min,7.100000,"8'29""",13h24,16:00 (+155min)
4,Vallon → Sixt,12.600000,782,841,2h 48min,4.500000,"13'21""",16h12,20:00 (+227min)
5,Sixt → Arrivée,12.100000,743,817,2h 44min,4.400000,"13'35""",18h57,23:30 (+273min)


## 8. Heures de passage vs barrières

In [10]:
def plot_passage_times(df_splits, barrieres, start_min):
    """Bar chart of predicted passage times vs cutoff times."""
    labels    = [r.split('→')[-1].strip() for r in df_splits['Tronçon']]
    t_passage = []
    for _, row in df_splits.iterrows():
        h_str = row['Heure arr.']
        t_passage.append(int(h_str[:2]) * 60 + int(h_str[3:]))

    labels_barr, t_barrieres = [], []
    for nom, dist, heure in barrieres:
        if heure is None:
            continue
        labels_barr.append(nom)
        t_barrieres.append(parse_hhmm(heure))

    def fmt(minutes):
        return f"{int(minutes)//60:02d}h{int(minutes)%60:02d}"

    tick_vals = list(range(int(min(t_passage + t_barrieres)) - 30,
                           int(max(t_passage + t_barrieres)) + 60, 60))

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=labels, y=t_passage,
        name='Heure de passage prédite', marker_color='#3498db',
        text=[fmt(t) for t in t_passage], textposition='outside',
    ))
    fig.add_trace(go.Scatter(
        x=labels_barr, y=t_barrieres,
        mode='markers+lines+text', name='Barrière horaire',
        marker=dict(color='#e74c3c', size=10, symbol='diamond'),
        line=dict(color='#e74c3c', dash='dash'),
        text=[fmt(t) for t in t_barrieres], textposition='top center',
    ))
    fig.update_layout(
        template='plotly_dark',
        title='Heures de passage prédites vs barrières horaires',
        xaxis=dict(title='Point de passage'),
        yaxis=dict(title='Heure', tickvals=tick_vals, ticktext=[fmt(v) for v in tick_vals]),
        height=450, margin=dict(t=60, b=50),
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
    )
    return fig


plot_passage_times(df_splits, BARRIERES, start_min).show()


## 9. Résumé global

In [11]:
end_min  = start_min + summary['total_min']
end_hhmm = f"{int(end_min)//60 % 24:02d}h{int(end_min)%60:02d}"

print('=' * 55)
print(f"  {summary['race_name']}")
print('=' * 55)
print(f"  Distance    : {summary['distance_km']} km")
print(f"  D+          : {summary['dplus_m']} m")
print(f"  D-          : {summary['dmoins_m']} m")
print(f"  Départ      : {START_TIME}")
print(f"  V_flat      : {V_FLAT_KMH} km/h  (facteur athlète {ATHLETE_FACTOR})")
print(f"  Fatigue     : a={FATIGUE_A}  k={FATIGUE_K}  actif={USE_FATIGUE}")
print(f"  Thermique   : seuil={T_THRESHOLD}°C  alpha={ALPHA_HEAT}  actif={USE_THERMAL}")
print(f"  Descente max: {V_DOWNHILL_MAX_KMH} km/h")
print(f"  Temps total : {summary['total_time_str']}")
print(f"  Allure moy. : {summary['avg_pace_str']} /km")
print(f"  Arrivée     : ~{end_hhmm}")
print('=' * 55)


  Trail du Mont-Blanc des Dames 2026
  Distance    : 71.28 km
  D+          : 4840 m
  D-          : 4855 m
  Départ      : 04:00
  V_flat      : 9 km/h  (facteur athlète 0.85)
  Fatigue     : a=0.753  k=2.981  actif=True
  Thermique   : seuil=15.0°C  alpha=0.004  actif=True
  Descente max: 9.0 km/h
  Temps total : 14h 57min
  Allure moy. : 12'35" /km
  Arrivée     : ~18h57


In [12]:
import json

# Extraire les durées en minutes depuis df_splits
segments_json = []
for _, row in df_splits.iterrows():
    # Convertir "Xh YYmin" ou "YYmin" en minutes
    temps = row['Temps']
    if 'h' in temps:
        parts = temps.replace('min', '').split('h')
        minutes = int(parts[0]) * 60 + int(parts[1].strip())
    else:
        minutes = int(temps.replace('min', '').strip())
    
    segments_json.append({
        'name': row['Tronçon'],
        'duration': minutes
    })

print("Segments à copier dans le widget :")
print(json.dumps(segments_json, ensure_ascii=False, indent=2))

Segments à copier dans le widget :
[
  {
    "name": "Départ → Joux Plane",
    "duration": 227
  },
  {
    "name": "Joux Plane → Golèse",
    "duration": 156
  },
  {
    "name": "Golèse → Refuge Bostan",
    "duration": 108
  },
  {
    "name": "Refuge Bostan → Vallon",
    "duration": 72
  },
  {
    "name": "Vallon → Sixt",
    "duration": 168
  },
  {
    "name": "Sixt → Arrivée",
    "duration": 164
  }
]


In [13]:
# from IPython.display import HTML

# with open('nutrition_planner.html', 'r') as f:
#     html_content = f.read()

# HTML(html_content)